In [32]:
import re
import requests
import geocoder
import time

In [33]:
url = 'http://127.0.0.1:1234/v1/chat/completions'

In [34]:
items = []

print('')

while True:
    item = input("welke gerechten heb je al/ heb je die je wilt gebruiken. (type stop om te stoppen): ")

    if item == "stop":
        break

    items.append(item)

In [35]:
g = geocoder.ip("me")
regio = g.city
lat = g.latlng[0]
lon = g.latlng[1]

In [36]:
prompts = [
    f"""
Geef exact 5 gerechten op basis van deze ingrediënten: {items}.

vaste opzet voor geen enkele reden dit aan passen:
gerecht nr|gerecht|ingredienten|duur|nieuw ingredient = (ingredient)

Voor elk gerecht:
- bereidingstijd <= 30 minuten
- extra ingrediënt
- en zet letterlijk dit aan het einde van de text zodat ik hem er later uit kan halen: 'nieuw ingredient = (ingredient)'
"""
]

In [37]:
for prompt in prompts:
    start_time = time.time()

In [38]:
data = {
    "model": "nvidia/nemotron-3-nano-4b",
    "messages": [
        {"role": "user", "content": prompt}
    ],
    "temperature": 0.7
}


In [39]:
try:
    response = requests.post(url, json=data)
    response.raise_for_status()

    end_time = time.time()

    result = response.json()

    answer = result["choices"][0]["message"]["content"]
    model_name = result.get("model", "onbekend")
    response_length = len(answer)

    print(answer)

    print("\nMetadata:")
    print(f"Modelnaam: {model_name}")
    print(f"Lengte response: {response_length} karakters")
    print(f"Tijd: {end_time - start_time:.2f} seconden")
    print("-" * 50)

except requests.exceptions.RequestException as e:
    print(f"Fout bij API-aanroep: {e}")


gerecht nr|gerecht|ingredienten|duur|nieuw ingrediënt = (ingredient)  
1|Kipkoek met sambal & ui|kip, brood, sambal, ui|20 min|nieuw ingrediënt = (kruiden)  
2|Kaas‑ en aardbeisalade met spes|kaas, aardbijen, spes, extra: geharde noten|25 min|nieuw ingrediënt = (geharde noten)  
3|Spek & ui grill met aardbies|spec, ui, aardbijen, citroensap|28 min|nieuw ingrediënt = (citreon)  
4|Sla‑ en broodje met kip en sambal|sla, brood, kip, sambal, extra: gehakte feta|30 min|nieuw ingrediënt = (gehakte feta)  
5|Kipperbroodje met ui en kaas|kip, ui, kaas, brood, extra: peterselie|27 min|nieuw ingrediënt = (peterselie)

Metadata:
Modelnaam: nvidia/nemotron-3-nano-4b
Lengte response: 615 karakters
Tijd: 15.73 seconden
--------------------------------------------------


In [42]:
for answe in answer.splitlines():
    match = re.search(r"nieuw ingrediënt\s*=\s*(.*?)", answe)

    if match:
        url = f"https://google.com/search?q=supermarkt+voor+{ingredient}+-ai"
        answe += f"|{url}"

    print(answe)



gerecht nr|gerecht|ingredienten|duur|nieuw ingrediënt = (ingredient)  |https://google.com/search?q=supermarkt+voor+peper+-ai
1|Kipkoek met sambal & ui|kip, brood, sambal, ui|20 min|nieuw ingrediënt = (kruiden)  |https://google.com/search?q=supermarkt+voor+peper+-ai
2|Kaas‑ en aardbeisalade met spes|kaas, aardbijen, spes, extra: geharde noten|25 min|nieuw ingrediënt = (geharde noten)  |https://google.com/search?q=supermarkt+voor+peper+-ai
3|Spek & ui grill met aardbies|spec, ui, aardbijen, citroensap|28 min|nieuw ingrediënt = (citreon)  |https://google.com/search?q=supermarkt+voor+peper+-ai
4|Sla‑ en broodje met kip en sambal|sla, brood, kip, sambal, extra: gehakte feta|30 min|nieuw ingrediënt = (gehakte feta)  |https://google.com/search?q=supermarkt+voor+peper+-ai
5|Kipperbroodje met ui en kaas|kip, ui, kaas, brood, extra: peterselie|27 min|nieuw ingrediënt = (peterselie)|https://google.com/search?q=supermarkt+voor+peper+-ai
